# Similarity-aware Label Smoothing

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset Preparation

In [7]:
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())

all_imgs = torch.stack([img for img, _ in train_ds], dim = 0)

ds_mean = all_imgs.mean(dim=(0, 2, 3))
ds_std = all_imgs.std(dim=(0, 2, 3))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((ds_mean, ), (ds_std, ))
])

test_ds = datasets.MNIST(root="./data", train=False, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

## Model

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

## Training Utils

### Accuracy Measure

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim = 1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

### Label Smoothing

In [ ]:
def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon / num_classes

## Training Loop

In [ ]:
def train(model, loader, optimizer, epochs = 10, epsilon = 0.1):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in tqdm(loader, leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x)

            targets = uniform_smooth_labels(y, num_classes=10, epsilon=epsilon)